# Results

In this notebook, we collect the tables and figures of the study from the saved runs in `experiments/results/`.
The runs are defined in `experiments/runs.py`, and each run is performed by `experiments/run.py`. Each error is the
relative $L^2$ error of the averaged validation controls against the exact equilibrium, for player 1 and player 2.

In [ ]:
import sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, '../src'); sys.path.insert(0, '../experiments')
import collect
from partial_info_games import use_style
use_style()
runs = collect.load()
CASES = {('two_traders', 0.0): 'two traders, $\\lambda = 0$', ('two_traders', 1.0): 'two traders, private, $\\lambda = 1$',
         ('markov_traders', 0.0): 'Markov chain, $\\lambda = 0$', ('broker_trader', np.nan): 'broker and trader',
         ('major_minor', np.nan): 'major and minor'}
def case(game, lam):
    return runs[(runs.game == game) & ((runs.lam == lam) if lam == lam else runs.lam.isna())]
def picard(frame):
    return frame[frame.control_variate & (frame.iterations == 16)]
print(len(runs), 'runs')

## 1. Accuracy against the number of paths

The errors are for seed 0 at 25,000 and 100,000 paths, and the mean and spread over seeds 0, 1 and 2 at 50,000 paths.

In [ ]:
rows = []
for (game, lam), label in CASES.items():
    frame = picard(case(game, lam))
    for paths, group in frame.groupby('paths'):
        errors = group[['error0', 'error1']].values * 100
        rows.append(dict(case=label, paths=paths, seeds=len(group), player1=f'{errors[:, 0].mean():.1f} ± {errors[:, 0].std():.1f}',
                         player2=f'{errors[:, 1].mean():.1f} ± {errors[:, 1].std():.1f}'))
pd.DataFrame(rows).set_index(['case', 'paths'])

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for (game, lam), label in CASES.items():
    frame = picard(case(game, lam)); summary = frame.assign(error=frame[['error0', 'error1']].mean(1)).groupby('paths').error
    ax.errorbar(summary.mean().index, summary.mean() * 100, yerr=summary.std().fillna(0) * 100, marker='o', capsize=3, label=label)
ax.set_xscale('log', base=2); ax.set_yscale('log'); ax.set_xlabel('training paths'); ax.set_ylabel('relative $L^2$ error (%)')
ax.legend(fontsize=9); plt.show()

## 2. Control variate

The errors are with and without the control variate, at 25,000 paths and seed 0.

In [ ]:
rows = []
for (game, lam), label in CASES.items():
    frame = case(game, lam); frame = frame[(frame.paths == 25000) & (frame.seed == 0) & (frame.iterations == 16)]
    for setting, mask in (('off', ~frame.control_variate), ('on', frame.control_variate)):
        if mask.any():
            row = frame[mask].iloc[0]; rows.append(dict(case=label, control_variate=setting, error=f"{row.error0 * 100:.1f} / {row.error1 * 100:.1f}"))
pd.DataFrame(rows).set_index(['case', 'control_variate'])

## 3. Markov-chain game at $\lambda = 1$

No exact equilibrium is known at $\lambda = 1$. We report the relative $L^2$ distance between the controls learned
from seeds 0, 1 and 2, and the first order condition residuals of each seed.

In [ ]:
frame = case('markov_traders', 1.0)
controls = [collect.controls(name) for name in frame.name]
distance = lambda a, b: np.sqrt(((a - b) ** 2).mean() / (b ** 2).mean())
print('relative L2 distance between seeds:', [round(distance(controls[a], controls[b]), 3) for a in range(len(controls)) for b in range(a + 1, len(controls))])
frame[['seed', 'residual0', 'residual1', 'mean_abs0', 'mean_abs1']].round(3)

## 4. Stronger coupling

The errors are for two traders with private signals at $\lambda = 1.5$ and $\lambda = 2$, with 25,000 paths and
seeds 0, 1 and 2, against the exact equilibrium.

In [ ]:
frame = picard(runs[(runs.game == 'two_traders') & runs.lam.isin([1.5, 2.0])])
frame.groupby('lam')[['error0', 'error1']].agg(['mean', 'std']).mul(100).round(1)